# Verify Causes of Duplicate (POS, new_ref, ALT_new) Cases

This notebook investigates why certain positions have duplicate entries in the new_vcf files.

Hypothesis: Duplicates occur when the original VCF has multiple variants at the same position (e.g., insertion + SNP, or two different SNPs).

In [1]:
import pandas as pd
import os
from pathlib import Path
from collections import defaultdict, Counter
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Define paths
base_dir = Path('../outputs/src_outputs')
new_vcf_dir = base_dir / 'new_vcf' / 'full_set'
original_vcf_dir = base_dir / 'modified_original_reference'

# Load the duplicate cases
duplicates_df = pd.read_csv('../outputs/scripts_outputs/duplicate_cases_detailed.csv')
print(f"Total duplicate cases to verify: {len(duplicates_df)}")
duplicates_df.head()

Total duplicate cases to verify: 704


,POS,new_ref,ALT_new,count,file_id
0,10882,C,G,2,ERR3243102
1,10882,C,G,2,ERR3241715
2,10882,C,G,2,ERR3243116
3,10882,C,G,2,ERR3239679
4,10882,C,G,2,ERR3239889


In [3]:
def classify_variant(ref, alt):
    """
    Classify a variant based on REF and ALT.
    Returns: 'SNP', 'insertion', 'deletion', 'complex'
    """
    ref_len = len(ref)
    alt_len = len(alt)
    
    if ref_len == 1 and alt_len == 1:
        return 'SNP'
    elif ref_len < alt_len and alt.startswith(ref):
        return 'insertion'
    elif ref_len > alt_len and ref.startswith(alt):
        return 'deletion'
    else:
        return 'complex'

In [4]:
def read_original_vcf(file_id):
    """
    Read the original VCF file for a given file_id.
    Returns DataFrame with columns: POS, REF, ALT, AF
    """
    vcf_path = original_vcf_dir / f"{file_id}_rDNA.vcf"
    
    if not vcf_path.exists():
        return None
    
    # Read VCF, skip header lines starting with #
    try:
        # First, find where the header ends
        with open(vcf_path, 'r') as f:
            for i, line in enumerate(f):
                if line.startswith('#CHROM'):
                    header_line = i
                    break
        
        df = pd.read_csv(vcf_path, sep='\t', skiprows=header_line)
        return df
    except Exception as e:
        print(f"Error reading {vcf_path}: {e}")
        return None

In [5]:
def analyze_duplicate_cause(file_id, pos):
    """
    For a given file and position, look up the original VCF to see
    what variants exist at that position.
    
    Returns dict with:
    - num_variants: how many variants at this position
    - variant_types: list of variant types (SNP, insertion, deletion, complex)
    - cause_category: categorization of what caused the duplicate
    - original_variants: list of (REF, ALT) tuples
    """
    vcf_df = read_original_vcf(file_id)
    
    if vcf_df is None:
        return {'error': 'VCF file not found'}
    
    # Find all variants at this position
    variants_at_pos = vcf_df[vcf_df['POS'] == pos]
    
    if len(variants_at_pos) == 0:
        return {'error': f'No variants found at position {pos}'}
    
    # Classify each variant
    variant_types = []
    original_variants = []
    
    for _, row in variants_at_pos.iterrows():
        ref = str(row['REF'])
        alt = str(row['ALT'])
        vtype = classify_variant(ref, alt)
        variant_types.append(vtype)
        original_variants.append((ref, alt, vtype))
    
    # Categorize the cause
    type_set = set(variant_types)
    
    if type_set == {'SNP'}:
        cause = 'multiple_SNPs'
    elif 'insertion' in type_set and 'SNP' in type_set:
        cause = 'insertion_plus_SNP'
    elif 'deletion' in type_set and 'SNP' in type_set:
        cause = 'deletion_plus_SNP'
    elif type_set == {'insertion'}:
        cause = 'multiple_insertions'
    elif type_set == {'deletion'}:
        cause = 'multiple_deletions'
    else:
        cause = 'other_combination'
    
    return {
        'num_variants': len(variants_at_pos),
        'variant_types': variant_types,
        'cause_category': cause,
        'original_variants': original_variants
    }

In [6]:
# Analyze all duplicate cases
results = []

print("Analyzing duplicate causes...")
for i, row in duplicates_df.iterrows():
    file_id = row['file_id']
    pos = row['POS']
    
    analysis = analyze_duplicate_cause(file_id, pos)
    
    result = {
        'file_id': file_id,
        'POS': pos,
        'new_ref': row['new_ref'],
        'ALT_new': row['ALT_new'],
        'dup_count': row['count']
    }
    result.update(analysis)
    results.append(result)
    
    if (i + 1) % 100 == 0:
        print(f"  Processed {i + 1}/{len(duplicates_df)} cases...")

print(f"\nDone! Analyzed {len(results)} duplicate cases.")

Analyzing duplicate causes...
  Processed 100/704 cases...
  Processed 200/704 cases...
  Processed 300/704 cases...
  Processed 400/704 cases...
  Processed 500/704 cases...
  Processed 600/704 cases...
  Processed 700/704 cases...

Done! Analyzed 704 duplicate cases.


In [7]:
# Create results DataFrame
results_df = pd.DataFrame(results)
results_df.head(20)

,file_id,POS,new_ref,ALT_new,dup_count,num_variants,variant_types,cause_category,original_variants,error
0,ERR3243102,10882,C,G,2,2.0,"[insertion, SNP]",insertion_plus_SNP,"[(G, GGC, insertion), (G, C, SNP)]",NaN
1,ERR3241715,10882,C,G,2,2.0,"[insertion, SNP]",insertion_plus_SNP,"[(G, GGC, insertion), (G, C, SNP)]",NaN
2,ERR3243116,10882,C,G,2,2.0,"[insertion, SNP]",insertion_plus_SNP,"[(G, GGC, insertion), (G, C, SNP)]",NaN
3,ERR3239679,10882,C,G,2,2.0,"[insertion, SNP]",insertion_plus_SNP,"[(G, GGC, insertion), (G, C, SNP)]",NaN
4,ERR3239889,10882,C,G,2,2.0,"[insertion, SNP]",insertion_plus_SNP,"[(G, GGC, insertion), (G, C, SNP)]",NaN
5,ERR3989026,7994,G,A,2,2.0,"[SNP, SNP]",multiple_SNPs,"[(A, C, SNP), (A, G, SNP)]",NaN
6,ERR3989032,7994,G,A,2,2.0,"[SNP, SNP]",multiple_SNPs,"[(A, C, SNP), (A, G, SNP)]",NaN
7,ERR3239490,10882,C,G,2,2.0,"[insertion, SNP]",insertion_plus_SNP,"[(G, GGC, insertion), (G, C, SNP)]",NaN
8,ERR3239490,12047,G,C,2,2.0,"[insertion, SNP]",insertion_plus_SNP,"[(C, CGCG, insertion), (C, G, SNP)]",NaN
9,ERR3988853,10882,C,G,2,2.0,"[insertion, SNP]",insertion_plus_SNP,"[(G, GGC, insertion), (G, C, SNP)]",NaN


## Summary: What Causes the Duplicates?

In [8]:
# Count by cause category
print("=" * 60)
print("DUPLICATE CAUSE BREAKDOWN")
print("=" * 60)

if 'cause_category' in results_df.columns:
    cause_counts = results_df['cause_category'].value_counts()
    print("\nCause Category Counts:")
    for cause, count in cause_counts.items():
        pct = 100 * count / len(results_df)
        print(f"  {cause}: {count} ({pct:.1f}%)")
else:
    print("Error column found - check for issues")
    print(results_df['error'].value_counts())

DUPLICATE CAUSE BREAKDOWN

Cause Category Counts:
  insertion_plus_SNP: 454 (64.5%)
  multiple_SNPs: 248 (35.2%)


In [9]:
# Breakdown by position - which positions have which cause
if 'cause_category' in results_df.columns:
    pos_cause = results_df.groupby(['POS', 'new_ref', 'ALT_new', 'cause_category']).size().reset_index(name='occurrences')
    pos_cause_sorted = pos_cause.sort_values('occurrences', ascending=False)
    
    print("\nMost common (POS, cause) combinations:")
    print(pos_cause_sorted.head(20).to_string(index=False))


Most common (POS, cause) combinations:
  POS new_ref ALT_new     cause_category  occurrences
10882       C       G insertion_plus_SNP          401
 7719       T       C      multiple_SNPs          152
 7994       G       A      multiple_SNPs           70
12047       G       C insertion_plus_SNP           26
10093       C       G insertion_plus_SNP           20
  673       C       G      multiple_SNPs           11
 7661       G       C      multiple_SNPs            3
  524       G       C insertion_plus_SNP            2
 3002       G       T      multiple_SNPs            2
 2392       C       G insertion_plus_SNP            2
  674       G       C      multiple_SNPs            2
 3090       C       G      multiple_SNPs            2
 3006       C       T      multiple_SNPs            1
 3090       C       G insertion_plus_SNP            1
  560       G       C      multiple_SNPs            1
 3635       C       G      multiple_SNPs            1
 2553       G       C      multiple_SNPs  

In [10]:
# Show examples of each cause category
if 'cause_category' in results_df.columns:
    print("\n" + "=" * 60)
    print("EXAMPLES OF EACH CAUSE CATEGORY")
    print("=" * 60)
    
    for cause in results_df['cause_category'].unique():
        print(f"\n--- {cause} ---")
        examples = results_df[results_df['cause_category'] == cause].head(3)
        for _, ex in examples.iterrows():
            print(f"  File: {ex['file_id']}, POS: {ex['POS']}")
            if 'original_variants' in ex and isinstance(ex['original_variants'], list):
                for ref, alt, vtype in ex['original_variants']:
                    print(f"    Original: {ref} -> {alt} ({vtype})")


EXAMPLES OF EACH CAUSE CATEGORY

--- insertion_plus_SNP ---
  File: ERR3243102, POS: 10882
    Original: G -> GGC (insertion)
    Original: G -> C (SNP)
  File: ERR3241715, POS: 10882
    Original: G -> GGC (insertion)
    Original: G -> C (SNP)
  File: ERR3243116, POS: 10882
    Original: G -> GGC (insertion)
    Original: G -> C (SNP)

--- multiple_SNPs ---
  File: ERR3989026, POS: 7994
    Original: A -> C (SNP)
    Original: A -> G (SNP)
  File: ERR3989032, POS: 7994
    Original: A -> C (SNP)
    Original: A -> G (SNP)
  File: ERR3239453, POS: 7994
    Original: A -> C (SNP)
    Original: A -> G (SNP)

--- nan ---


In [11]:
# Check if ALL duplicates are accounted for by multiple variants at same position
if 'num_variants' in results_df.columns:
    print("\n" + "=" * 60)
    print("VERIFICATION: Are all duplicates from multiple variants?")
    print("=" * 60)
    
    multi_variant = (results_df['num_variants'] >= 2).sum()
    single_variant = (results_df['num_variants'] == 1).sum()
    missing = results_df['num_variants'].isna().sum()
    
    print(f"\nCases with 2+ variants at same position: {multi_variant} ({100*multi_variant/len(results_df):.1f}%)")
    print(f"Cases with only 1 variant at position: {single_variant}")
    print(f"Cases with missing data: {missing}")
    
    if single_variant > 0:
        print("\nUnexpected single-variant cases:")
        print(results_df[results_df['num_variants'] == 1][['file_id', 'POS', 'new_ref', 'ALT_new']])


VERIFICATION: Are all duplicates from multiple variants?

Cases with 2+ variants at same position: 702 (99.7%)
Cases with only 1 variant at position: 0
Cases with missing data: 2


## Conclusion

The analysis above shows exactly why duplicates occur and categorizes them.

In [12]:
# Save the verification results
output_path = Path('../outputs/scripts_outputs/duplicate_causes_verified.csv')

# Create a clean version for export (convert list columns to strings)
export_df = results_df.copy()
if 'original_variants' in export_df.columns:
    export_df['original_variants'] = export_df['original_variants'].apply(
        lambda x: str(x) if isinstance(x, list) else x
    )
if 'variant_types' in export_df.columns:
    export_df['variant_types'] = export_df['variant_types'].apply(
        lambda x: str(x) if isinstance(x, list) else x
    )

export_df.to_csv(output_path, index=False)
print(f"Saved verification results to: {output_path}")

Saved verification results to: ../outputs/scripts_outputs/duplicate_causes_verified.csv
